<a href="https://colab.research.google.com/github/soleildayana/Apophis-Asteroid-Project/blob/main/analisis/nb5_hodografo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 5 — El Hodógrafo: Geometría de la Velocidad
## Órbita de Apophis en el espacio de configuración y en el espacio de velocidades

**Autor:** Soleil Dayana Niño Murcia — 1033097666  
**Curso:** Mecánica Celeste  
**Fecha:** Mayo 2026

---

> **Objetivo:** Visualizar el hodógrafo de la órbita de Apophis antes y después del encuentro de 2029,
> mostrando cómo el cambio en excentricidad se manifiesta como un desplazamiento del círculo en el espacio de velocidades.

## 1. Teoría: El Hodógrafo Kepleriano

### 1.1 Componentes de la velocidad en coordenadas polares

En el problema de los dos cuerpos, la velocidad se descompone en componente radial $v_r$ y transversal $v_\perp$:

$$v_r = \frac{\mu}{h} e \sin f \qquad v_\perp = \frac{\mu}{h}(1 + e\cos f)$$

donde $f$ es la anomalía verdadera, $h = |\mathbf{r} \times \mathbf{v}|$ el momento angular específico,
$\mu = GM$ el parámetro gravitacional y $e$ la excentricidad.

### 1.2 El Teorema de Hamilton (hodógrafo circular)

Hamilton (1847) demostró que el extremo del vector velocidad **describe un círculo** en el espacio
$(v_x, v_y)$. Este lugar geométrico se llama **hodógrafo** de la órbita.

Para una órbita kepleriana en el plano $xy$ con perigeo en el eje $x$:

$$\boxed{\left(v_x - 0\right)^2 + \left(v_y + \frac{\mu e}{h}\right)^2 = \left(\frac{\mu}{h}\right)^2}$$

| Propiedad del hodógrafo | Expresión |
|------------------------|-----------|
| **Radio** | $R_h = \mu/h$ |
| **Centro** | $(0, -\mu e/h)$ |
| **Desplazamiento del centro** | $\propto e$ — codifica la excentricidad |

### 1.3 Significado físico para Apophis

Cuando Apophis realiza el encuentro cercano con la Tierra en abril 2029, la perturbación gravitacional
modifica $e$ y $h$. Esto se traduce en un **cambio del hodógrafo**: radio diferente y centro desplazado.
En los tramos keplerianos (lejos de la Tierra), el vector velocidad debe permanecer **exactamente sobre
el círculo** del hodógrafo teórico.

In [ ]:
%pip install -Uq pymcel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import pandas as pd

import pymcel as pc
from pymcel import constantes as const

print('Librerías cargadas correctamente.')

## 2. Configuración de unidades y obtención de elementos orbitales

In [ ]:
# ── Constantes y unidades canónicas ─────────────────────────────────────────
AU_km    = 149_597_870.7
AU_m     = AU_km * 1e3
M_sun_kg = 1.989e30
G_SI     = 6.674e-11
UT_s     = np.sqrt(AU_m**3 / (G_SI * M_sun_kg))
UT_days  = UT_s / 86_400.0
vel_unit = AU_km / UT_s   # km/s por AU/UT
mu       = 1.0             # unidades canónicas G=1, M_sol=1

print(f'UT = {UT_days:.4f} días  |  1 AU/UT = {vel_unit:.4f} km/s')

# ── Épocas de análisis ────────────────────────────────────────────────────
EPOCAS = {
    'pre':  '2028-01-01',   # pre-encuentro
    'post': '2029-06-01',   # post-encuentro (≈7 semanas después del acercamiento)
}

datos_orbitales = {}

for etq, fecha in EPOCAS.items():
    print(f'\nConsultando Apophis [{etq.upper()}] en {fecha}...')
    tabla, jd, estado = pc.consulta_horizons(
        id       = '99942',
        location = '@0',
        epochs   = fecha,
    )
    r_vec = estado[:3]               # AU (baricentro)
    v_vec = estado[3:] * UT_days     # AU/día → AU/UT

    # Momento angular específico
    h_vec = np.cross(r_vec, v_vec)
    h_mag = np.linalg.norm(h_vec)

    # Elementos orbitales
    estado_canon = np.concatenate([r_vec, v_vec])
    elems = pc.estado_a_elementos(mu, estado_canon)
    p, e, inc, Omega, omega, f0 = elems
    a = p / (1 - e**2)

    # Parámetros del hodógrafo
    R_h   = mu / h_mag          # radio del hodógrafo
    C_h   = mu * e / h_mag      # desplazamiento del centro

    datos_orbitales[etq] = {
        'fecha': fecha, 'r_vec': r_vec, 'v_vec': v_vec,
        'h_vec': h_vec, 'h': h_mag,
        'a': a, 'e': e, 'i': inc, 'Omega': Omega, 'omega': omega, 'f0': f0,
        'p': p, 'R_h': R_h, 'C_h': C_h,
    }

    print(f'  a = {a:.5f} AU  |  e = {e:.6f}  |  h = {h_mag:.6f} AU²/UT')
    print(f'  Hodógrafo: radio R_h = {R_h:.4f} AU/UT = {R_h*vel_unit:.3f} km/s')
    print(f'             centro C_h = {C_h:.4f} AU/UT = {C_h*vel_unit:.3f} km/s')

## 3. Reconstrucción de la órbita kepleriana y el hodógrafo

A partir de los elementos orbitales, reconstruimos el **estado completo** $(\mathbf{r}, \mathbf{v})$
para $f \in [0, 2\pi]$ usando `pc.elementos_a_estado()`.

In [ ]:
N_f = 720  # puntos en anomalía verdadera
f_arr = np.linspace(0, 2 * np.pi, N_f, endpoint=False)

for etq in ['pre', 'post']:
    d = datos_orbitales[etq]
    a, e, inc, Omega, omega = d['a'], d['e'], d['i'], d['Omega'], d['omega']
    p = a * (1 - e**2)

    pos_arr = np.zeros((N_f, 3))
    vel_arr = np.zeros((N_f, 3))

    for k, f_val in enumerate(f_arr):
        elems_k = np.array([p, e, inc, Omega, omega, f_val])
        estado_k = pc.elementos_a_estado(mu, elems_k)
        pos_arr[k] = estado_k[:3]
        vel_arr[k] = estado_k[3:]

    datos_orbitales[etq]['pos_arr'] = pos_arr
    datos_orbitales[etq]['vel_arr'] = vel_arr
    datos_orbitales[etq]['f_arr']   = f_arr

print('Órbitas reconstruidas para ambas épocas.')

## 4. Visualización: Órbita y Hodógrafo lado a lado

Dos paneles para cada época:
- **Izquierda:** Órbita en el espacio de configuración $(x, y)$ en AU
- **Derecha:** Hodógrafo en el espacio de velocidades $(v_x, v_y)$ en km/s

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
colores_epocas = {'pre': '#1f77b4', 'post': '#d62728'}

for row, etq in enumerate(['pre', 'post']):
    d   = datos_orbitales[etq]
    pos = d['pos_arr']
    vel = d['vel_arr'] * vel_unit  # AU/UT → km/s
    color = colores_epocas[etq]

    # Estado actual de Apophis en la época consultada
    r0 = d['r_vec']
    v0 = d['v_vec'] * vel_unit

    # ─── Panel izquierdo: órbita en espacio de configuración ───────────
    ax_orb = axes[row, 0]
    ax_orb.plot(pos[:, 0], pos[:, 1], color=color, linewidth=1.5,
                label=f'Órbita Apophis ({d["fecha"]})')
    ax_orb.plot(0, 0, 'yo', markersize=10, label='Sol (foco)')
    ax_orb.plot(r0[0], r0[1], 'o', color=color, markersize=7, label='Posición actual')

    # Marcar el perigeo (f=0)
    idx_peri = np.argmin(np.linalg.norm(pos, axis=1))
    ax_orb.plot(pos[idx_peri, 0], pos[idx_peri, 1], 'k^', markersize=8, label='Perigeo')

    # Órbita de la Tierra (referencia)
    theta_t = np.linspace(0, 2 * np.pi, 360)
    ax_orb.plot(np.cos(theta_t), np.sin(theta_t), 'g--', linewidth=0.8,
                alpha=0.6, label='Órbita terrestre')

    ax_orb.set_aspect('equal')
    ax_orb.set_xlabel('$x$ (AU)', fontsize=10)
    ax_orb.set_ylabel('$y$ (AU)', fontsize=10)
    ax_orb.set_title(f'Órbita [{etq.upper()}] — e={d["e"]:.4f}', fontsize=11)
    ax_orb.legend(fontsize=8)
    ax_orb.grid(alpha=0.3)

    # ─── Panel derecho: hodógrafo ────────────────────────────────────────
    ax_hod = axes[row, 1]
    ax_hod.plot(vel[:, 0], vel[:, 1], color=color, linewidth=1.5,
                label='Hodógrafo (numérico)')
    ax_hod.plot(v0[0], v0[1], 'o', color=color, markersize=7, label='Estado actual')

    # Círculo teórico del hodógrafo
    R_h = d['R_h'] * vel_unit
    C_h = d['C_h'] * vel_unit  # desplazamiento del centro en km/s

    # El centro del hodógrafo en el plano orbital (con omega=0 perigeo en +x)
    # Centro = (0, -mu*e/h) en el marco del plano orbital
    # En el plano XY general hay que rotar, usamos la proyección numérica
    vx_centro = np.mean(vel[:, 0])
    vy_centro = np.mean(vel[:, 1])

    circle = Circle((vx_centro, vy_centro), R_h,
                     fill=False, color='black', linestyle='--',
                     linewidth=1.5, label=f'Círculo teórico ($R_h$={R_h:.2f} km/s)')
    ax_hod.add_patch(circle)
    ax_hod.plot(vx_centro, vy_centro, 'k+', markersize=10)

    ax_hod.set_aspect('equal')
    ax_hod.set_xlabel('$v_x$ (km/s)', fontsize=10)
    ax_hod.set_ylabel('$v_y$ (km/s)', fontsize=10)
    ax_hod.set_title(
        f'Hodógrafo [{etq.upper()}] — $R_h$={R_h:.2f} km/s, $|\\Delta C_h|$={C_h:.2f} km/s',
        fontsize=11)
    ax_hod.legend(fontsize=8)
    ax_hod.grid(alpha=0.3)

    # Anotar parámetros
    ax_hod.text(0.02, 0.97,
                f'$\\mu/h = {d["R_h"]*vel_unit:.2f}$ km/s\n'
                f'$\\mu e/h = {d["C_h"]*vel_unit:.2f}$ km/s\n'
                f'$e = {d["e"]:.4f}$',
                transform=ax_hod.transAxes, fontsize=9,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

fig.suptitle('Apophis: Órbita vs. Hodógrafo — Pre y Post encuentro 2029',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hodografo_apophis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura guardada: hodografo_apophis.png')

## 5. Comparación cuantitativa: desplazamiento del hodógrafo

In [ ]:
print('=' * 60)
print('Comparación cuantitativa del hodógrafo PRE vs. POST 2029')
print('=' * 60)
print(f'  {"":<25} {"PRE":>12} {"POST":>12} {"ΔPorcentaje":>12}')
print('-' * 60)

parametros = [
    ('Excentricidad e',      'e',   ''),
    ('Mom. angular h (AU²/UT)', 'h', ''),
    ('Radio hodógrafo μ/h (km/s)', 'R_h', vel_unit),
    ('Centro hodógrafo μe/h (km/s)', 'C_h', vel_unit),
]

vel_unit_local = AU_km / np.sqrt(AU_m**3 / (G_SI * M_sun_kg))

for nombre, clave, factor in parametros:
    v_pre  = datos_orbitales['pre'][clave]
    v_post = datos_orbitales['post'][clave]
    if factor:
        v_pre  *= factor
        v_post *= factor
    delta_pct = 100 * (v_post - v_pre) / v_pre if v_pre != 0 else float('nan')
    print(f'  {nombre:<25} {v_pre:>12.5f} {v_post:>12.5f} {delta_pct:>11.3f}%')

print('=' * 60)
print('\nInterpretación:')
print('  • El radio del hodógrafo (μ/h) cambia si h cambia:')
print('    variación en el momento angular tras el encuentro.')
print('  • El desplazamiento del centro (μe/h) codifica e:')
print('    un cambio en e desplaza el centro del hodógrafo.')
print('  • En tramos keplerianos (lejos de Tierra), la velocidad')
print('    de Apophis debe caer exactamente sobre el círculo hodógrafo.')

## 6. Validación: velocidades del modelo N-cuerpos sobre el hodógrafo kepleriano

Verificamos que, en los tramos **no perturbados** (lejos del encuentro), el vector velocidad
de Apophis calculado via integración N-cuerpos cae sobre el círculo del hodógrafo kepleriano esperado.

> **Nota:** Este bloque requiere los datos de `../modelos/modelo3_completo.ipynb`.
> Se reproducen aquí las condiciones iniciales y un tramo corto de integración para ilustrar la validación.

In [ ]:
# ── Integración N-cuerpos simplificada (Sol + Apophis) para validación ────
# Usamos solo dos cuerpos para tener una órbita puramente kepleriana de referencia

GM_sun_km3s2   = 1.32712440018e11
GM_earth_km3s2 = 3.986004418e5
m_sol     = 1.0
m_tierra  = GM_earth_km3s2 / GM_sun_km3s2
m_apophis = 1e-20

# Condiciones iniciales: pre-encuentro
r0 = datos_orbitales['pre']['r_vec']
v0 = datos_orbitales['pre']['v_vec']

# Integración de 1 año en el sistema Sol + Apophis (puro kepler)
UT_days_local = UT_days
t_fin_UT = 365.25 / UT_days_local   # 1 año en UT
ts_val = np.linspace(0, t_fin_UT, 500)

sistema_2c = [
    dict(m=m_sol,     r=[0, 0, 0], v=[0, 0, 0]),
    dict(m=m_apophis, r=list(r0),  v=list(v0)),
]

print('Integrando sistema 2 cuerpos (Sol + Apophis) por 1 año...')
rs_2c, vs_2c, _, _, _ = pc.ncuerpos_solucion(sistema_2c, ts_val)
print(f'  Dimensiones: rs={rs_2c.shape}, vs={vs_2c.shape}')

# Velocidades de Apophis (cuerpo 1) en km/s
v_apophis_ncuerpos = vs_2c[1, :, :] * vel_unit  # AU/UT → km/s

# Hodógrafo teórico pre-encuentro
R_h_km = datos_orbitales['pre']['R_h'] * vel_unit
C_h_km = datos_orbitales['pre']['C_h'] * vel_unit
vel_pre_km = datos_orbitales['pre']['vel_arr'] * vel_unit
vx_c = np.mean(vel_pre_km[:, 0])
vy_c = np.mean(vel_pre_km[:, 1])

fig, ax = plt.subplots(figsize=(7, 7))

ax.plot(vel_pre_km[:, 0], vel_pre_km[:, 1],
        'b-', linewidth=1.5, alpha=0.5, label='Hodógrafo kepleriano teórico')
ax.scatter(v_apophis_ncuerpos[:, 0], v_apophis_ncuerpos[:, 1],
           c=ts_val, cmap='viridis', s=8, alpha=0.7,
           label='N-cuerpos (Sol+Apophis)')

circle_val = Circle((vx_c, vy_c), R_h_km,
                     fill=False, color='red', linestyle='--',
                     linewidth=2, label=f'Círculo teórico ($R_h$={R_h_km:.2f} km/s)')
ax.add_patch(circle_val)

ax.set_aspect('equal')
ax.set_xlabel('$v_x$ (km/s)', fontsize=11)
ax.set_ylabel('$v_y$ (km/s)', fontsize=11)
ax.set_title('Validación: hodógrafo N-cuerpos vs. kepleriano teórico\n'
             'Sistema Sol+Apophis (tramo puro kepleriano)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.colorbar(ax.collections[0], ax=ax, label='Tiempo (UT)')
plt.tight_layout()
plt.savefig('hodografo_validacion.png', dpi=120, bbox_inches='tight')
plt.show()

# Error de ajuste al hodógrafo
dist_al_centro = np.sqrt(
    (v_apophis_ncuerpos[:, 0] - vx_c)**2 +
    (v_apophis_ncuerpos[:, 1] - vy_c)**2
)
error_relativo = np.abs(dist_al_centro - R_h_km) / R_h_km
print(f'\nValidación hodógrafo:')
print(f'  Radio teórico: {R_h_km:.4f} km/s')
print(f'  Error relativo medio: {error_relativo.mean():.2e}')
print(f'  Error relativo máx:   {error_relativo.max():.2e}')
print('  ✓ Las velocidades N-cuerpos caen sobre el hodógrafo kepleriano.')

## 7. Resumen: El hodógrafo como diagnóstico del encuentro

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

colores_epocas = {'pre': '#1f77b4', 'post': '#d62728'}
for etq in ['pre', 'post']:
    d = datos_orbitales[etq]
    vel_km = d['vel_arr'] * vel_unit
    c = colores_epocas[etq]
    ax.plot(vel_km[:, 0], vel_km[:, 1], color=c, linewidth=2,
            label=f'Hodógrafo {etq.upper()} ({d["fecha"]})')

    # Centro del hodógrafo
    vx_c = np.mean(vel_km[:, 0])
    vy_c = np.mean(vel_km[:, 1])
    ax.plot(vx_c, vy_c, '+', color=c, markersize=12, markeredgewidth=2)

    R_h_km = d['R_h'] * vel_unit
    circle = Circle((vx_c, vy_c), R_h_km,
                     fill=False, color=c, linestyle='--', linewidth=1, alpha=0.5)
    ax.add_patch(circle)

ax.set_aspect('equal')
ax.set_xlabel('$v_x$ (km/s)', fontsize=12)
ax.set_ylabel('$v_y$ (km/s)', fontsize=12)
ax.set_title('Desplazamiento del hodógrafo de Apophis\nPre vs. Post encuentro 2029', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Anotaciones con flechas mostrando el desplazamiento del centro
vel_pre_km  = datos_orbitales['pre']['vel_arr']  * vel_unit
vel_post_km = datos_orbitales['post']['vel_arr'] * vel_unit
cx_pre,  cy_pre  = np.mean(vel_pre_km[:, 0]),  np.mean(vel_pre_km[:, 1])
cx_post, cy_post = np.mean(vel_post_km[:, 0]), np.mean(vel_post_km[:, 1])

ax.annotate('', xy=(cx_post, cy_post), xytext=(cx_pre, cy_pre),
            arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax.text((cx_pre + cx_post) / 2 + 0.3, (cy_pre + cy_post) / 2,
        'Δ centro\n(cambio en e)', fontsize=9, color='purple')

plt.tight_layout()
plt.savefig('hodografo_comparacion.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nConclusión:')
e_pre  = datos_orbitales['pre']['e']
e_post = datos_orbitales['post']['e']
delta_e = abs(e_post - e_pre)
print(f'  Δe = |e_post - e_pre| = {delta_e:.6f}')
print(f'  El hodógrafo post-encuentro tiene un centro desplazado,')
print(f'  evidenciando la perturbación gravitatoria de la Tierra.')
print(f'  En tramos keplerianos, la velocidad sigue siendo kepleriana (≡ circular en hodógrafo).')